# SPARC Pipeline Demo

This notebook demonstrates the basic usage of the SPARC (Spectral Pattern Analysis for ROI Classification) pipeline. 

**Prerequisites:**
1. Ensure you have downloaded the SAM model weights (`sam_vit_h_4b8939.pth`).
2. Ensure you have the required external repositories installed (`asdf`, `pdr`, etc.).
3. Set the `IOF_DATA_PATH` variable below to point to your data directory.

In [5]:
import sys
import logging
from pathlib import Path

# Import SPARC
from src.sparc import Sparc, export_spectra_csv, configure_logging

# Configure logging to see detailed progress
configure_logging(verbose=True)

In [6]:
# === CONFIGURE THESE PATHS ===
SOL = "0525"
IOF_DATA_PATH = f"e:/products/{SOL}/iof"
SAM_MODEL_PATH = "C:\\Users\\Lars\\Documents\\Python\\mars_lab\\models\\sam_vit_h_4b8939.pth"

# Optional: Sequence ID and observation index
SEQ_ID = None  # e.g., "ZR1_0613"
OBS_IX = 4

print(f"IOF Path: {IOF_DATA_PATH}")
print(f"SAM Model: {SAM_MODEL_PATH}")

IOF Path: e:/products/0525/iof
SAM Model: C:\Users\Lars\Documents\Python\mars_lab\models\sam_vit_h_4b8939.pth


## 1. Initialize Pipeline

We initialize the `Sparc` class with the path to the SAM model. We enable GPU acceleration and threading for performance.

In [7]:
sparc = Sparc(
    sam_model_path=SAM_MODEL_PATH,
    use_gpu=True,       # Will fall back to CPU if CUDA is not available
    use_threading=True, # Enables parallel ROI extraction
    verbose=True        # Print debug stats
)

21:20:09 - src.sparc.core.sparc - DEBUG - SPARC initialized with verbose logging enabled


## 2. Run Processing Steps

We can chain the methods together to run the full pipeline. 

* **Load:** Reads IOF data and aligns stereo cameras.
* **Preprocess:** Masks shadows/sky and converts to R*.
* **Segment:** Uses SAM to find semantic regions.
* **Extract ROIs:** Sub-clusters segments to find homogeneous spectral regions.
* **Analyze:** Detects outliers and clusters spectra.
* **Select:** Picks the best representative ROI for each cluster.

In [8]:
(
    sparc
    .load(
        iof_path=IOF_DATA_PATH, 
        seq_id=SEQ_ID, 
        obs_ix=OBS_IX
    )
    .preprocess(
        apply_r_star=True
    )
    .segment(
        points_per_side=32,   # Higher = more detailed segmentation
        pred_iou_thresh=0.88
    )
    .extract_rois(
        area_threshold=50,       # Minimum ROI size in pixels
        min_cluster_area=500,    # Min segment size to attempt sub-clustering
        min_clean_area=4000      # Min size after cleaning artifacts
    )
    .analyze(
        contamination=0.1,    # Estimated % of outliers in data
        max_components=None   # Auto-detect number of spectral clusters
    )
    .select()                 # Final heuristic selection
)

21:20:09 - src.sparc.core.pipeline - INFO - Loading data from e:/products/0525/iof
21:20:13 - src.sparc.core.pipeline - INFO - Loaded scene: SOL0525_zcam03420_RSM3098
21:20:13 - src.sparc.core.pipeline - DEBUG - Data loading took 4.40s
21:20:13 - src.sparc.core.pipeline - DEBUG - Merged cube shape: (14, 1178, 1598)
21:20:13 - src.sparc.core.pipeline - DEBUG - Left cube shape: (9, 1178, 1598)
21:20:13 - src.sparc.core.pipeline - DEBUG - Right cube shape: (9, 1178, 1598)
21:20:13 - src.sparc.core.pipeline - INFO - Preprocessing data (masking and calibration)
21:20:18 - src.sparc.core.pipeline - DEBUG - Mask Stats: Shadow=9.7%, Sky=0.0%
21:20:18 - src.sparc.core.pipeline - DEBUG - Valid pixels remaining: 85.7%
21:20:18 - src.sparc.core.pipeline - DEBUG - Calibrated Data Range: [0.000, 1.344]
21:20:18 - src.sparc.core.pipeline - DEBUG - Mean Reflectance: 0.312
21:20:18 - src.sparc.core.pipeline - DEBUG - Preprocessing took 4.83s
21:20:18 - src.sparc.core.pipeline - INFO - Segmenting image 

## 3. Visualization

Plot the summary of the run, showing the RGB context, the segmentation map, the final selected ROIs, and their corresponding spectra.

In [ ]:
sparc.plot(figsize=(16, 12))

## 4. Accessing Data Programmatically

You can access the raw data directly from the result object.

In [11]:
result = sparc.result

print(f"Scene ID: {result.scene_id}")
print(f"Number of final ROIs: {len(result.final_rois)}")
print(f"Number of spectral clusters: {result.n_clusters}")

# Example: Print coordinates of the first ROI
if len(result.final_rois) > 0:
    x, y, w, h = result.final_rois[0]
    print(f"First ROI: x={x}, y={y}, w={w}, h={h}")

Scene ID: SOL0525_zcam03420_RSM3098
Number of final ROIs: 9
Number of spectral clusters: 9
First ROI: x=1276, y=1120, w=13, h=13
